<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/gone_by_the_plans(14).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# Cible : deux hyperplans parallèles {P_sep = 0} et {P_sep = ε} (P_sep linéaire),
# label sigmoïde séparant les deux plans. Méthode : experts cylindriques
# (gaussiennes anisotropes) + Wendland anisotropes, UN σ aléatoire par dimension
# et par centre, SANS décroissance géométrique de σ.
# ══════════════════════════════════════════════════════════════════════════════
params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20,
    "seeds": {1: 22476, 2: 12425, 3: 12421, 42: 48124, 43: 24322, 44: 12561, 11: 21365409, 12: 2335},
    "weights":   {0: 0.01, 1: 1, 2: 0.1, 3: 0.0},
    "domain":    (0.0, 10.0),  # domaine [lo, hi]^d où sont tirés les points
    "epsilon":   0.7,     # écartement des deux plans (valeur de P_sep, pas distance)
    "n_train":   10,
    "n_test":    5000,
    "n_levels":     0,     # nb d'experts cylindriques (gaussiennes anisotropes)
    "n_levels_wnd": 0,     # nb d'experts Wendland anisotropes (support compact)
    "k_loss":    3.0,      # sélection : garder les experts avec loss <= k_loss * meilleure
    "deg_poly_expert": 9, # -1 -> expert polynomial désactivé
    "n_dict":    5000,     # taille du dictionnaire par niveau
    "n_centres": 800,      # centres retenus par niveau
    "sigma_min": 0.2,      # borne inf des sigma
    "sigma_max": 1,      # borne sup des sigma
    "sigma_mode": "aligned",  # "random" : σ anisotrope tiré au hasard (ancien) |
                              # "aligned": pour chaque centre, on tire N orientations
                              #   σ sur ‖σ‖=1 et on GARDE celle de norme H minimale
                              #   (aligne l'atome sur la variété), puis l'échelle est
                              #   tirée à part -> découvre la direction normale locale.
    "n_orient":  200,      # nb d'orientations testées par centre (mode aligned)
    "candidate_select": "random",  # "random" : sous-échantillon des candidats,
                                   #   indépendant des labels (la pénalité Sobolev
                                   #   trie) | "corr" : ancien top-k par corrélation.
    "lambda_reg": 1e3,    # UNIQUE régularisation Sobolev (Phase 1, sélection, Phase 2)

    "a_label":   1.0,   # amplitude des labels ±a_label (plateaux haut/bas)
    "beta":      1e-3,     # raideur log-cosh ; garder beta*a_label ~ O(1)
    "loss_type": "qp",# "sigmoid" (tanh bornée [0,1], NON convexe) |
                           # "qp" (min ‖u‖_H s.c. marge dure, CONVEXE, stable) |
                           # "hinge2" | "logcosh" | "mse"
    "qp_margin": 1.0,      # marge dure a du QP : y_i u(x_i) >= a
    "beta_sig":  1.0,      # raideur DIRECTE de la sigmoïde : transition sur f ~ ±1/beta.
                           # f est une combinaison de noyaux O(1) -> beta d'ordre 1.
                           # (a_label ne sert plus qu'à hinge2)
    "n_G":       2000,     # points pour la Gram de chaque f_i
    "n_G_H":     5000,     # points pour la Gram finale sur H
    "thres1": 1e-30,
    "thres2": 1
}
params["n_unlabeled"]        = 4000 - params["n_train"]
params["train_center_ratio"] = 0    # tous les centres tirés côté train+unlabeled

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:,j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:,j]**e
                y += c * term
            return y
        return Pn
    return P

d   = params["n_ambiant"]
eps = params["epsilon"]

# P_sep LINÉAIRE (degré 1) -> les deux surfaces {P_sep=0} et {P_sep=ε} sont de
# vrais hyperplans parallèles, ce qui permet une projection EXACTE (cf. plus bas).
P_sep, idx_sep, coeffs_sep = random_sparse_polynomial(d, 1, 4, seed=99)
P_sep = normalize_polynomial(P_sep, idx_sep, coeffs_sep)

# ══════════════════════════════════════════════════════════════════════════════
# CLOUD sur {Q=0} = {P_sep=0} ∪ {P_sep=ε}   —  PROJECTION EXACTE
#
# BUG CORRIGÉ : l'ancienne version projetait par descente de gradient normalisée
# sur Q = P_sep·(P_sep−ε). Deux défauts :
#   (1) ∇Q = (2·P_sep − ε)·∇P_sep s'annule sur le plan médian {P_sep=ε/2} :
#       les points initialisés près de ce plan restaient PIÉGÉS (gradient nul).
#   (2) Taux de convergence effondré (~0.5 %) -> la boucle de sur-échantillonnage
#       redessinait sans fin et retenait surtout les points du plan le plus
#       facile à atteindre, d'où un déséquilibre ~70/30 entre les deux plans.
#
# Comme P_sep est AFFINE, P_sep(x) = a·x + b, la projection orthogonale d'un
# point x sur l'hyperplan {a·x + b = t} est analytique :
#       x' = x − ((a·x + b − t)/‖a‖²) · a
# On échantillonne dans le cube [lo,hi]^d, on projette exactement, on garde les
# points restés dans le cube. On tire 50/50 entre les deux plans (équilibre exact).
# ══════════════════════════════════════════════════════════════════════════════
DOM_LO, DOM_HI = params["domain"]

_b = P_sep(np.zeros((1, d)))[0]
_a = np.array([P_sep(np.eye(d)[k:k+1])[0] - _b for k in range(d)])
_a2 = float(_a @ _a)
assert _a2 > 0, "P_sep dégénéré (gradient nul)"

def project_to_plane(X, t):
    """Projection orthogonale exacte de X sur l'hyperplan {a·x + b = t}."""
    lam = (X @ _a + _b - t) / _a2
    return X - lam[:, None] * _a

def sample_on_plane(t, n_target, seed, max_tries=1000000):
    """Points uniformes (par rejet) sur {P_sep=t} ∩ [lo,hi]^d, projection exacte."""
    rng = np.random.default_rng(seed); out = []; got = 0; tries = 0
    while got < n_target and tries < max_tries:
        m = max(4 * (n_target - got), 500)
        X  = rng.uniform(DOM_LO, DOM_HI, (m, d))
        Xp = project_to_plane(X, t)
        ok = np.all((Xp >= DOM_LO) & (Xp <= DOM_HI), axis=1)
        g  = Xp[ok]
        if len(g): out.append(g); got += len(g)
        tries += m
    if got < n_target:
        raise RuntimeError(f"plan t={t}: seulement {got}/{n_target} points collectés")
    return np.vstack(out)[:n_target]

def sample_two_planes(n_target, seed):
    """n_target points répartis 50/50 EXACTEMENT sur les deux plans, mélangés."""
    n0 = n_target // 2; n1 = n_target - n0
    X0 = sample_on_plane(0.0, n0, seed)
    X1 = sample_on_plane(eps, n1, seed + 1)
    X  = np.vstack([X0, X1])
    rng = np.random.default_rng(seed + 2)
    perm = rng.permutation(len(X))
    return X[perm]

print("Génération du cloud sur {Q=0} = {P_sep=0} ∪ {P_sep=ε} (projection exacte, 50/50)...")
X_train     = sample_two_planes(params["n_train"],     seed=params["seeds"][42])
X_test      = sample_two_planes(params["n_test"],      seed=params["seeds"][43])
X_unlabeled = sample_two_planes(params["n_unlabeled"], seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_test:{X_test.shape}  X_unlabeled:{X_unlabeled.shape}")

# contrôle d'équilibre + résidu de projection
for nom, Xc in [("train", X_train), ("test", X_test)]:
    p = P_sep(Xc)
    n0 = int((np.abs(p) < eps/2).sum()); n1 = int((np.abs(p-eps) < eps/2).sum())
    resid = np.minimum(np.abs(p), np.abs(p-eps)).max()
    print(f"  {nom:5s} : plan0={n0}  planε={n1}  max|résidu projection|={resid:.2e}")

# ── Cible : label ±a_label selon le plan (plateau haut / plateau bas) ──────────
# CLASSIFICATION : y = +a_label sur {P_sep=ε}, −a_label sur {P_sep=0}. La loss
# log-cosh sature loin de la frontière : seul le côté du seuil (0) compte.
A_LABEL   = params["a_label"]     # marge de hinge2 (échelle de f)
BETA_SIG  = params["beta_sig"]    # raideur directe de la sigmoïde (labels ±1)
def target_label(X):
    return np.where(P_sep(X) > eps/2, A_LABEL, -A_LABEL)

y_train = target_label(X_train); y_test = target_label(X_test)
# Labels ±1 pour les BASELINES (RBF, Ridge poly) : leur échelle naturelle.
# Imposer ±a_label aux baselines biaiserait la comparaison (leur régularisation
# interne est calibrée pour des cibles O(1)). La décision étant sign(f), l'échelle
# du label est un paramètre interne à chaque méthode -> chacun sur la sienne.
y_train_pm1 = np.sign(y_train); y_test_pm1 = np.sign(y_test)

# ── Échelle de travail de la MÉTHODE H ────────────────────────────────────────
# sigmoid : f est ajusté sur des labels ±1 (β règle la marge sur l'échelle de f),
#           donc TOUT le pipeline H (fit, sélection, affichage, MSE) est en ±1.
# hinge2/logcosh/mse : la marge/cible est a_label, pipeline en ±a_label.
if params["loss_type"] in ("sigmoid", "qp"):
    yH_train, yH_test = y_train_pm1, y_test_pm1
else:
    yH_train, yH_test = y_train, y_test
n_pos = int((y_train > 0).sum()); n_neg = int((y_train < 0).sum())
print(f"  y_train: {n_neg} × (−{A_LABEL:g})  |  {n_pos} × (+{A_LABEL:g})   "
      f"(labels ±a_label, décision sign(f))")

X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

# ══════════════════════════════════════════════════════════════════════════════
# ÉCHANTILLONNAGE DES CENTRES ET SIGMAS (anisotrope, UN σ aléatoire par dim/centre,
# log-uniforme dans [sigma_min, sigma_max], SANS décroissance)
# ══════════════════════════════════════════════════════════════════════════════
def _structure_matrix(deriv_fn, c, X_local, weights, d):
    """Matrice de structure S (d×d) : moyenne locale des produits de dérivées
    directionnelles, pondérée par les poids Sobolev. Sa direction propre MINIMALE
    est celle où l'atome varie le moins sur le cloud (= le long de la variété) ;
    sa direction propre MAXIMALE est la normale à la variété.

    Construction : on évalue les dérivées d'un atome ISOTROPE de référence
    (σ=1 partout) en c sur X_local, et on forme S à partir du gradient (ordre 1)
    et de la hessienne (ordre 2), qui encodent comment φ varie selon chaque axe.
    Pour des plans, S a un spectre : une grande valeur propre (direction normale a)
    et d−1 petites (directions tangentes)."""
    sig1 = np.ones((1, d))
    phi, grad, hess, third = deriv_fn(X_local, c[None,:], sig1, weights)
    S = np.zeros((d, d))
    w1 = weights.get(1,0.)
    if w1 and grad is not None:
        g = grad[:,0,:]                      # (nloc, d)
        S += w1 * (g.T @ g) / len(g)
    w2 = weights.get(2,0.)
    if w2 and hess is not None:
        h = hess[:,0,:,:]                    # (nloc, d, d)
        # contribution d'ordre 2 : moyenne de H·Hᵀ contractée -> (d,d)
        S += w2 * np.einsum('xik,xjk->ij', h, h) / len(h)
    return S

def sample_candidates_analytic(X_train, X_unlabeled, n_candidates, train_ratio,
                               rng, deriv_fn, X_cloud, weights, n_local=200):
    """Orientation optimale ANALYTIQUE par centre : σ aligné sur le vecteur propre
    MINIMAL de la matrice de structure (direction de moindre variation sur le
    cloud). Pas de tirage -> exact pour des plans (trouve la normale a). L'échelle
    ‖σ‖ reste libre (log-uniforme). σ anisotrope : petit dans la direction normale
    (résolution fine ⟂ variété), grand dans les directions tangentes."""
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates - n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    nC = len(centers); dloc = X_cloud.shape[1]
    scales = np.exp(rng.uniform(np.log(params["sigma_min"]),
                                np.log(params["sigma_max"]), size=nC))
    # ratio d'anisotropie : σ_normal / σ_tangent (petit = très allongé sur la variété)
    aniso = params.get("aniso_ratio", 0.1)
    sigmas = np.empty((nC, dloc))
    for i in range(nC):
        c = centers[i]
        d2 = np.sum((X_cloud - c)**2, axis=1)
        loc = X_cloud[np.argsort(d2)[:min(n_local, len(X_cloud))]]
        S = _structure_matrix(deriv_fn, c, loc, weights, dloc)
        evals, evecs = np.linalg.eigh(S)      # croissant : evecs[:,-1] = normale
        # σ : petit (aniso·scale) dans la direction normale (grande valeur propre),
        #     grand (scale) dans les d−1 directions tangentes.
        inv_widths = np.ones(dloc)
        inv_widths[-1] = aniso                # dernière = direction normale
        sig_axes = scales[i] * inv_widths     # échelle par axe propre
        # reconstruire σ dans la base canonique : σ_k = sqrt(Σ_j evecs[k,j]² sig_axes[j]²)
        sigmas[i] = np.sqrt((evecs**2) @ (sig_axes**2))
    return centers, sigmas

def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates - n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    log_s = rng.uniform(np.log(params["sigma_min"]), np.log(params["sigma_max"]),
                        size=(len(centers), d))
    sigmas = np.exp(log_s)
    return centers, sigmas

def _atom_Hnorm2(deriv_fn, c, sigma, X_local, weights):
    """Norme H² d'UN atome (centre c, forme sigma) mesurée sur X_local.
    Réutilise les dérivées analytiques ; renvoie Σ_o w_o mean_x ||D^o phi||²."""
    phi, grad, hess, third = deriv_fn(X_local, c[None,:], sigma[None,:], weights)
    tot = 0.0
    w0 = weights.get(0,0.)
    if w0: tot += w0 * np.mean(phi[:,0]**2)
    w1 = weights.get(1,0.)
    if w1 and grad  is not None: tot += w1 * np.mean(np.sum(grad[:,0,:]**2,  axis=1))
    w2 = weights.get(2,0.)
    if w2 and hess  is not None: tot += w2 * np.mean(np.sum(hess[:,0,:,:]**2, axis=(1,2)))
    w3 = weights.get(3,0.)
    if w3 and third is not None: tot += w3 * np.mean(np.sum(third[:,0,:,:,:]**2, axis=(1,2,3)))
    return tot

def sample_candidates_aligned(X_train, X_unlabeled, n_candidates, train_ratio,
                              rng, deriv_fn, X_cloud, weights, n_orient,
                              n_local=200):
    """Pour CHAQUE centre : tire n_orient orientations σ sur ‖σ‖=1, garde celle
    de norme H MINIMALE (atome le plus 'plat' sur le cloud -> aligné sur la
    variété). L'échelle ‖σ‖ est ensuite tirée log-uniforme dans [σmin,σmax].

    Le minimum de norme H à ‖σ‖=1 fixée sélectionne l'orientation dont les
    dérivées sont les plus faibles SUR LE CLOUD : un ellipsoïde allongé le long
    de la variété (quasi constant sur le cloud) plutôt que normal à elle.
    """
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates - n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    nC = len(centers)

    # échelles (rayons) libres, tirées comme avant mais SCALAIRES par centre
    scales = np.exp(rng.uniform(np.log(params["sigma_min"]),
                                np.log(params["sigma_max"]), size=nC))

    sigmas = np.empty((nC, X_cloud.shape[1]))
    for i in range(nC):
        c = centers[i]
        # points du cloud proches de c (l'atome ‖σ‖~1 ne voit que le voisinage)
        d2 = np.sum((X_cloud - c)**2, axis=1)
        loc = X_cloud[np.argsort(d2)[:min(n_local, len(X_cloud))]]
        # n_orient orientations sur la sphère ‖σ‖=1 (σ>0 : demi-sphère positive)
        O = np.abs(rng.normal(size=(n_orient, X_cloud.shape[1])))
        O /= np.linalg.norm(O, axis=1, keepdims=True)
        best_h = np.inf; best_o = O[0]
        for o in O:
            h = _atom_Hnorm2(deriv_fn, c, o, loc, weights)   # à ‖σ‖=1
            if h < best_h:
                best_h = h; best_o = o
        sigmas[i] = scales[i] * best_o        # orientation optimale × échelle libre
    return centers, sigmas

# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES CYLINDRIQUES (anisotropes) : features + dérivées analytiques
# ══════════════════════════════════════════════════════════════════════════════
def cyl_features(X, centers, sigmas):
    diff = X[:,None,:] - centers[None,:,:]
    sq_w = np.sum(diff**2 / sigmas[None,:,:]**2, axis=2)
    return np.exp(-sq_w)

def cyl_derivatives(X, centers, sigmas, weights=None):
    d_ = X.shape[1]
    diff = X[:,None,:] - centers[None,:,:]
    inv_s2 = 1./sigmas**2
    diff_w = diff * inv_s2[None,:,:]
    sq_w = np.sum(diff*diff_w, axis=2)
    phi = np.exp(-sq_w)
    grad = -2*diff_w*phi[:,:,None]
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id = np.eye(d_)
        diag  = -2*inv_s2[None,:,:,None]*Id[None,None,:,:]
        cross = 4*np.einsum('nmi,nmj->nmij', diff_w, diff_w)
        hess  = (diag+cross)*phi[:,:,None,None]
    else: hess = None
    if need_third:
        Id = np.eye(d_)
        inv_s2_bc = inv_s2[None,:,:]
        cubic = -8*np.einsum('nmi,nmj,nmk->nmijk', diff_w, diff_w, diff_w)
        sym   = 4*(np.einsum('ij,nmi,nmk->nmijk', Id, inv_s2_bc, diff_w)+
                   np.einsum('ik,nmi,nmj->nmijk', Id, inv_s2_bc, diff_w)+
                   np.einsum('jk,nmj,nmi->nmijk', Id, inv_s2_bc, diff_w))
        third = (cubic+sym)*phi[:,:,None,None,None]
    else: third = None
    return phi, grad, hess, third

# ══════════════════════════════════════════════════════════════════════════════
# WENDLAND ANISOTROPES : support compact {r<1}, r²=Σ (x_k−c_k)²/σ_k².
# Profil C^{2k} minimal tel que 2k ≥ ordre max des poids (ordre 3 -> C⁴).
# ══════════════════════════════════════════════════════════════════════════════
def _wnd_profile(max_order):
    k = max(1, int(np.ceil(max_order/2)))
    sd = lambda num, r: np.where(r > 1e-12, num/np.maximum(r, 1e-12), 0.0)
    if k == 1:      # C²
        return (lambda r,rp: rp**4*(4*r+1),
                lambda r,rp: -20*rp**3,
                lambda r,rp: sd(60*rp**2, r),
                None)
    elif k == 2:    # C⁴
        return (lambda r,rp: rp**6*(35*r**2+18*r+3)/3,
                lambda r,rp: -(56/3)*rp**5*(5*r+1),
                lambda r,rp: 560.*rp**4,
                lambda r,rp: sd(-2240.*rp**3, r))
    else:           # C⁶
        return (lambda r,rp: rp**8*(32*r**3+25*r**2+8*r+1),
                lambda r,rp: -22*rp**7*(16*r**2+7*r+1),
                lambda r,rp: 528*rp**6*(6*r+1),
                lambda r,rp: -22176.*rp**5)

_wnd_max_order = max([o for o, w in params["weights"].items() if w != 0])
WND_PHI, WND_H1, WND_H2, WND_H3 = _wnd_profile(_wnd_max_order)

def wnd_features(X, centers, sigmas):
    u = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    return WND_PHI(r, rp)

def wnd_derivatives(X, centers, sigmas, weights=None):
    d_ = X.shape[1]
    u  = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r  = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    inv_s  = 1./sigmas
    inv_s2 = inv_s**2
    us  = u*inv_s[None,:,:]
    phi = WND_PHI(r, rp)
    H1  = WND_H1(r, rp)
    grad = H1[:,:,None]*us
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id = np.eye(d_)
        H2 = WND_H2(r, rp)
        hess = (H2[:,:,None,None]*np.einsum('nmi,nmj->nmij', us, us)
                + H1[:,:,None,None]*(Id[None,None,:,:]*inv_s2[None,:,:,None]))
    else: hess = None
    if need_third:
        assert WND_H3 is not None, "poids d'ordre 3 : profil Wendland C⁴ minimum requis"
        Id = np.eye(d_)
        H3 = WND_H3(r, rp)
        cubic = H3[:,:,None,None,None]*np.einsum('nmi,nmj,nmk->nmijk', us, us, us)
        sym = (np.einsum('ij,mi,nmk->nmijk', Id, inv_s2, us)
              +np.einsum('ik,mi,nmj->nmijk', Id, inv_s2, us)
              +np.einsum('jk,mj,nmi->nmijk', Id, inv_s2, us))
        third = cubic + WND_H2(r, rp)[:,:,None,None,None]*sym
    else: third = None
    return phi, grad, hess, third

def build_G(phi, grad, hess, third, weights, n):
    G = np.zeros((phi.shape[1], phi.shape[1]))
    w0 = weights.get(0,0.)
    if w0: G += w0*(phi.T@phi)/n
    w1 = weights.get(1,0.)
    if w1 and grad  is not None: G += w1*np.einsum('xik,xjk->ij',     grad, grad,  optimize='optimal')/n
    w2 = weights.get(2,0.)
    if w2 and hess  is not None: G += w2*np.einsum('xikl,xjkl->ij',   hess, hess,  optimize='optimal')/n
    w3 = weights.get(3,0.)
    if w3 and third is not None: G += w3*np.einsum('xiklm,xjklm->ij', third,third, optimize='optimal')/n
    return G

def sobolev_norm2(derivs, weights, idx=None):
    """||f||²_H = Σ_o w_o · mean_x ||D^o f(x)||²_F pour une fonction déjà évaluée."""
    tot = 0.
    for key, order, axes in (('phi',0,()), ('grad',1,(1,)),
                             ('hess',2,(1,2)), ('third',3,(1,2,3))):
        w = weights.get(order, 0.)
        if not w:            continue
        T = derivs.get(key)
        if T is None:        continue
        if idx is not None:  T = T[idx]
        tot += w * (np.mean(T**2) if order == 0
                    else np.mean(np.sum(T**2, axis=axes)))
    return tot

# ══════════════════════════════════════════════════════════════════════════════
# SOLVEUR log-cosh régularisé Sobolev  (Newton amorti / IRLS)
#
#   min_c  (1/n) Σ_i logcosh( β (y_i − (A c)_i) )  +  λ  cᵀ G c
#
# logcosh'(z) = tanh(z),  logcosh''(z) = sech²(z) = 1 − tanh²(z).
# Résidu r = y − A c. Avec u = β r :
#   grad_c = −(β/n) Aᵀ tanh(u)            + 2 λ G c
#   Hess_c =  (β²/n) Aᵀ diag(sech²(u)) A  + 2 λ G
# La Hessienne a la structure AᵀWA + 2λG : on la résout par eigh, exactement
# comme la forme close des moindres carrés (W = I dans ce cas). Newton amorti
# avec back-tracking sur la vraie loss -> pas de divergence même si β·a_label≫1.
# ══════════════════════════════════════════════════════════════════════════════

def _logcosh_loss(A, c, y, beta, G, lam):
    u = beta * (y - A @ c)
    # logcosh stable : |u| + softplus(−2|u|) − log2  = log cosh(u)
    au = np.abs(u)
    lc = au + np.log1p(np.exp(-2*au)) - np.log(2.0)
    data = np.mean(lc) / (beta**2)      # normalisation ∝ 1/β² -> échelle MSE quand β→0
    reg  = lam * (c @ G @ c)
    return data + reg, data, reg

def solve_logcosh(A, y, G, lam, beta, c0=None, n_newton=50, tol=1e-9, verbose=False):
    """Newton amorti pour la loss log-cosh régularisée Sobolev.
    A:(n,k)  y:(n,)  G:(k,k)  ->  c:(k,), rang effectif."""
    n, k = A.shape
    c = np.zeros(k) if c0 is None else c0.copy()
    loss_prev, _, _ = _logcosh_loss(A, c, y, beta, G, lam)
    rank_eff = k
    for it in range(n_newton):
        u  = beta * (y - A @ c)
        th = np.tanh(u)
        w  = 1.0 - th**2                      # sech²(u) ∈ (0,1]
        # gradient et Hessienne (échelle 1/β² pour rester comparable au MSE)
        grad = -(1.0/(beta*n)) * (A.T @ th) + 2*lam*(G @ c)
        AW   = A * w[:, None]
        Hess = (1.0/n) * (A.T @ AW) + 2*lam*G
        # résolution eigh (comme la forme close), troncature spectrale
        ev, evec = np.linalg.eigh(Hess)
        thresh = max(lam, 1e-12) * 1e-6
        m = ev > thresh
        rank_eff = int(m.sum())
        if rank_eff == 0:
            break
        step = evec[:, m] @ ((evec[:, m].T @ (-grad)) / ev[m])
        # back-tracking : garantit la décroissance de la vraie loss
        t = 1.0
        for _ in range(40):
            c_new = c + t * step
            loss_new, _, _ = _logcosh_loss(A, c_new, y, beta, G, lam)
            if loss_new <= loss_prev - 1e-4 * t * (grad @ step):
                break
            t *= 0.5
        if verbose:
            print(f"      newton {it:2d} | loss={loss_new:.6e} | t={t:.3g} | rang={rank_eff}")
        if loss_prev - loss_new < tol * max(1.0, abs(loss_prev)):
            c = c_new; loss_prev = loss_new; break
        c = c_new; loss_prev = loss_new
    return c, rank_eff

# ══════════════════════════════════════════════════════════════════════════════
# SOLVEUR hinge quadratique régularisé Sobolev  (Newton / IRLS)
#
#   min_c  (1/n) Σ_i max(0, m − y_i (A c)_i)²  +  λ cᵀ G c
#
# m = marge = a_label. Pour le point i, soit s_i = y_i (A c)_i (score signé).
#   • s_i ≥ m  -> point GAGNÉ : perte 0, gradient 0, hessienne 0 (il sort du système)
#   • s_i < m  -> point ACTIF : perte (m − s_i)², contribue.
# Résidu de marge  e_i = max(0, m − s_i).  Avec masque actif a_i = [s_i < m] :
#   grad_c = −(2/n) Aᵀ (y ⊙ e)            + 2 λ G c
#   Hess_c =  (2/n) Aᵀ diag(a) A          + 2 λ G      (y_i²=1)
# Structure AᵀWA + 2λG, W = diag(a) binaire : c'est la forme close, résolue par
# eigh. « Gagné = ne contribue plus » est EXACT (a_i = 0). Newton amorti avec
# back-tracking sur la vraie loss -> convergence garantie.
# ══════════════════════════════════════════════════════════════════════════════

def _hinge2_loss(A, c, y, margin, G, lam):
    s = y * (A @ c)                       # score signé
    e = np.maximum(0.0, margin - s)       # résidu de marge
    data = np.mean(e**2) / (margin**2)    # ∝ 1/m² -> échelle stable vs a_label
    reg  = lam * (c @ G @ c)
    return data + reg, data, reg

def solve_hinge2(A, y, G, lam, margin, c0=None, n_newton=100, tol=1e-10, verbose=False):
    """Newton/IRLS pour la hinge quadratique régularisée Sobolev.
    A:(n,k)  y:(n,) ∈ {±?}  G:(k,k). Retourne (c, n_actifs, rang)."""
    n, k = A.shape
    c = np.zeros(k) if c0 is None else c0.copy()
    m2 = margin**2
    loss_prev, _, _ = _hinge2_loss(A, c, y, margin, G, lam)
    n_active = n; rank_eff = k
    # ── Active-set / IRLS : à masque fixé, la hinge² est un moindres carrés
    #    régularisé Sobolev -> solution exacte par eigh. On alterne :
    #    (1) résoudre exactement sur les points actifs, (2) réévaluer le masque.
    #    Un back-tracking ne sert que de garde-fou si le masque oscille.
    for it in range(n_newton):
        s   = y * (A @ c)
        act = s < margin                  # points ACTIFS (marge non atteinte)
        n_active = int(act.sum())
        if n_active == 0:                 # tout le monde a gagné : loss data = 0
            rank_eff = 0; break
        Aa = A[act]; ya = y[act]
        # sous-problème exact : min_c (1/(m²n))||m·ya − Aa c||² + λ cᵀGc
        # (car sur les actifs, ya·(Aa c) < m, l'objectif hinge² = (m − ya·Aa c)²)
        M   = (Aa.T @ Aa)/(m2*n) + lam*G
        rhs = (Aa.T @ (margin*ya))/(m2*n)
        ev, evec = np.linalg.eigh(M)
        thresh = max(lam, 1e-12) * 1e-6
        msk = ev > thresh
        rank_eff = int(msk.sum())
        if rank_eff == 0: break
        c_target = evec[:, msk] @ ((evec[:, msk].T @ rhs) / ev[msk])
        # pas complet vers la solution exacte du sous-problème ; back-tracking
        # seulement si la vraie loss (masque réévalué) n'a pas baissé.
        step = c_target - c
        t = 1.0
        for _ in range(30):
            c_new = c + t*step
            loss_new, _, _ = _hinge2_loss(A, c_new, y, margin, G, lam)
            if loss_new <= loss_prev + 1e-12:
                break
            t *= 0.5
        improved = loss_prev - loss_new
        c = c + t*step; loss_prev = loss_new
        if verbose:
            print(f"      irls {it:2d} | loss={loss_new:.6e} | t={t:.3g} | "
                  f"actifs={n_active}/{n} | rang={rank_eff}")
        if improved < tol * max(1.0, abs(loss_prev)):
            break
    return c, n_active, rank_eff

# ══════════════════════════════════════════════════════════════════════════════
# SOLVEUR perte sigmoïde bornée (tanh) régularisée Sobolev  (Newton amorti)
#
#   min_c  (1/n) Σ_i ℓ( y_i, (A c)_i )  +  λ cᵀ G c ,
#   ℓ(y,f) = (1 − tanh(β y f)) / 2 ∈ [0,1].
#
# Propriétés voulues :
#   • bornée : ℓ ∈ [0,1] toujours (aucun point ne peut dominer la somme) ;
#   • ℓ → 0  quand y f → +∞ (bien classé), zéro ASYMPTOTIQUE ;
#   • ℓ → 1  quand y f → −∞ (mal classé), saturé ;
#   • β calé sur a/2 : β = C_BETA / (a/2), transition ~ [−a/2, a/2].
# NON CONVEXE (sature des deux côtés) -> Newton peut voir des minima locaux ;
# on amortit par back-tracking et on peut multi-démarrer.
#
# Dérivées (z = β y f) :
#   ℓ  = (1 − tanh z)/2
#   dℓ/df = −(β y /2) sech² z
#   d²ℓ/df² = β² sech² z · tanh z         (change de signe -> non convexe)
# Pour Newton on utilise une Hessienne de Gauss-Newton PSD (on garde la partie
# positive), ce qui stabilise : W_i = (β²/2) sech²(z_i)|tanh(z_i)|_+ ... ->
# en pratique W_i = (β²/2) sech²(z_i) (approx PSD, régularisée par λG).
# ══════════════════════════════════════════════════════════════════════════════

def _sigm_loss(A, c, y, beta, G, lam):
    z  = beta * (y * (A @ c))
    l  = 0.5 * (1.0 - np.tanh(z))
    data = np.mean(l)
    reg  = lam * (c @ G @ c)
    return data + reg, data, reg

def solve_sigmoid(A, y, G, lam, beta, c0=None, n_newton=200, tol=1e-11,
                  verbose=False):
    """Newton amorti (Gauss-Newton PSD) pour la perte tanh bornée + Sobolev.
    A:(n,k)  y:(n,)  G:(k,k). Retourne (c, n_marge, rang).
    n_marge = nb de points encore dans la zone de transition (|z|<3)."""
    n, k = A.shape
    c = np.zeros(k) if c0 is None else c0.copy()
    loss_prev, _, _ = _sigm_loss(A, c, y, beta, G, lam)
    rank_eff = k; n_margin = n
    for it in range(n_newton):
        f  = A @ c
        z  = beta * (y * f)
        sech2 = 1.0 / np.cosh(z)**2
        th = np.tanh(z)
        # gradient exact
        grad = -(beta/(2.0*n)) * (A.T @ (y * sech2)) + 2*lam*(G @ c)
        # Hessienne Gauss-Newton PSD : (β²/2n) Aᵀ diag(sech²·(1+|th|... )) A
        # on prend le poids positif w = (β²/2) sech²·(1 − th·... ) -> simplifier :
        # w_i = (β²/2) sech²(z_i) ; c'est la courbure au voisinage de la frontière,
        # PSD, et régularisée par 2λG (toujours définie positive).
        w = (beta**2/2.0) * sech2
        AW = A * w[:, None]
        Hess = (1.0/n) * (A.T @ AW) + 2*lam*G
        ev, evec = np.linalg.eigh(Hess)
        thresh = max(lam, 1e-12) * 1e-6
        msk = ev > thresh
        rank_eff = int(msk.sum())
        if rank_eff == 0: break
        step = evec[:, msk] @ ((evec[:, msk].T @ (-grad)) / ev[msk])
        # back-tracking sur la vraie loss (non convexe -> on exige décroissance)
        t = 1.0; gTs = grad @ step
        ok = False
        for _ in range(50):
            c_new = c + t*step
            loss_new, _, _ = _sigm_loss(A, c_new, y, beta, G, lam)
            if loss_new <= loss_prev + 1e-4 * t * gTs:
                ok = True; break
            t *= 0.5
        if not ok:               # aucune décroissance -> minimum local atteint
            break
        improved = loss_prev - loss_new
        c = c_new; loss_prev = loss_new
        n_margin = int((np.abs(z) < 3).sum())
        if verbose:
            print(f"      sigm {it:2d} | loss={loss_new:.6e} | t={t:.3g} | "
                  f"marge={n_margin}/{n} | rang={rank_eff}")
        if improved < tol * max(1.0, abs(loss_prev)):
            break
    return c, n_margin, rank_eff

# ══════════════════════════════════════════════════════════════════════════════
# MÉTRIQUES COHÉRENTES avec la loss active
# Pour l'affichage : on ne mesure PAS une MSE quand la méthode n'optimise pas la
# MSE. On renvoie (data, reg) = la VRAIE loss de la méthode, plus l'accuracy
# (fraction de points du bon côté de 0), la seule métrique interprétable en
# classification. f est le vecteur des prédictions déjà calculées.
# ══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
# SOLVEUR QP : semi-interpolation Sobolev sur le polyèdre de marge
#
#   min_c  cᵀ G c    s.c.   y_i (A c)_i >= a   pour tout i du train
#
# = projection, au sens de la norme H, sur le convexe E = {u : y_i u(x_i) >= a}.
# Forme quadratique CONVEXE sur un polyèdre (n contraintes = n_train, très peu).
# Contrairement aux loss (sigmoïde non convexe, etc.), la solution est UNIQUE et
# STABLE : pas de minimum parasite, pas de fuite. Si le plateau (norme H ~0) est
# dans le span des atomes, le QP le trouve. Si E est vide (dico incapable de
# séparer avec marge a), le QP est INFAISABLE -> diagnostic net.
# ══════════════════════════════════════════════════════════════════════════════
from scipy.optimize import minimize, LinearConstraint

def solve_qp(A, y, G, margin, verbose=False):
    """min cᵀGc s.c. diag(y)Ac >= margin. Retourne (c, faisable, rang_effectif)."""
    n, k = A.shape
    Gr = G + 1e-10 * np.eye(k)            # définie positive
    C  = np.diag(y) @ A                    # contraintes y_i (Ac)_i >= margin
    con = LinearConstraint(C, lb=margin, ub=np.inf)
    # point de départ admissible : moindres carrés vers (marge·y) amplifié
    try:
        c0 = np.linalg.lstsq(A, 1.5*margin*y, rcond=None)[0]
    except Exception:
        c0 = np.zeros(k)
    res = minimize(lambda c: c@Gr@c, c0, jac=lambda c: 2*Gr@c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y * (A @ c)))
    faisable  = marge_eff >= margin - 1e-4
    if verbose:
        print(f"      QP: succès={res.success} marge_atteinte={marge_eff:.3f} "
              f"||u||_H={np.sqrt(max(c@G@c,0)):.4f}")
    # rang effectif : nb de contraintes actives (à la marge)
    rank_eff = int(np.sum(np.abs(y*(A@c) - margin) < 1e-3))
    return c, faisable, max(rank_eff, 1)

def method_metrics(f_train, f_test, coeffs, G, lam, yH_tr, yH_te):
    lt = params["loss_type"]
    # data-term dans l'échelle propre de la loss (comparable entre experts)
    if lt == "sigmoid":
        z = BETA_SIG * (yH_tr * f_train)
        data = np.mean(0.5*(1.0 - np.tanh(z)))            # ∈ [0,1]
    elif lt == "hinge2":
        s = yH_tr * f_train
        data = np.mean(np.maximum(0.0, params["a_label"] - s)**2) / params["a_label"]**2
    elif lt == "logcosh":
        u = params["beta"] * (yH_tr - f_train)
        au = np.abs(u)
        data = np.mean(au + np.log1p(np.exp(-2*au)) - np.log(2.0)) / params["beta"]**2
    else:  # mse
        data = np.mean((yH_tr - f_train)**2)
    reg = lam * (coeffs @ G @ coeffs) if coeffs is not None else np.nan
    # sign(0)=0 ne matche aucun label ±1 -> une fonction nulle afficherait 0.000
    # (trompeur : c'est du hasard, pas 0% de bonnes réponses). On compte 0.5.
    def _acc(f, y):
        s = np.sign(f)
        return np.mean(np.where(s == 0, 0.5, (s == np.sign(y)).astype(float)))
    acc_tr = _acc(f_train, yH_tr)
    acc_te = _acc(f_test,  yH_te)
    return data, reg, acc_tr, acc_te

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts cylindriques + Wendland indépendants
# Chaque f_i = argmin sur son propre dictionnaire aléatoire (σ sans decay)
# ══════════════════════════════════════════════════════════════════════════════
level_specs = ([("cyl", i) for i in range(params["n_levels"])] +
               [("wnd", i) for i in range(params["n_levels_wnd"])])
n_experts_total = len(level_specs)
expert_names    = [f"{typ}_{i+1:02d}" for typ, i in level_specs]

print(f"\nPhase 1 : calcul de {n_experts_total} experts "
      f"({params['n_levels']} cyl + {params['n_levels_wnd']} "
      f"wnd C{2*max(1,int(np.ceil(_wnd_max_order/2)))})...")

F_train = np.zeros((len(X_train), n_experts_total))
F_test  = np.zeros((len(X_test),  n_experts_total))
F_all   = np.zeros((N_all,        n_experts_total))
fi_derivs_list = []
times_p1 = []

for slot, (lev_type, lev_idx) in enumerate(level_specs):
    t0 = time.time()
    rng = np.random.default_rng(seed={"cyl":0, "wnd":2000}[lev_type]+lev_idx)

    feat_fn  = wnd_features    if lev_type=="wnd" else cyl_features
    deriv_fn = wnd_derivatives if lev_type=="wnd" else cyl_derivatives

    if params["sigma_mode"] == "aligned":
        candidates, sigmas_cand = sample_candidates_aligned(
            X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"],
            rng, deriv_fn, X_all, params["weights"], params["n_orient"])
    else:
        candidates, sigmas_cand = sample_candidates(
            X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)

    A_cand = feat_fn(X_train, candidates, sigmas_cand)

    # ── SÉLECTION DES CENTRES ─────────────────────────────────────────────────
    # candidate_select = "corr" : ancien filtre par corrélation avec y (top-k).
    #   Sur 10 pts binaires, ce critère ne garde que les atomes collés aux points
    #   d'entraînement -> fabrique du sur-apprentissage AVANT le fit. Obsolète ici.
    # candidate_select = "random" : sous-échantillon aléatoire, INDÉPENDANT de y.
    #   On laisse la pénalité Sobolev λ‖f‖²_G du fit faire le tri.
    k = min(params["n_centres"], len(candidates))
    if params.get("candidate_select", "random") == "corr":
        scores = ((A_cand.T @ y_train) / len(X_train))**2
        idx_sel = np.argsort(scores)[-k:]
    else:
        idx_sel = rng.choice(len(candidates), size=k, replace=False)

    centers_sel = candidates[idx_sel]
    sigmas_sel  = sigmas_cand[idx_sel]
    A     = A_cand[:, idx_sel]
    Atest = feat_fn(X_test, centers_sel, sigmas_sel)
    Aall  = feat_fn(X_all,  centers_sel, sigmas_sel)

    phi, grad, hess, third = deriv_fn(X_all, centers_sel, sigmas_sel, params["weights"])

    n_G   = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G(phi[idx_G],
                grad[idx_G]  if grad  is not None else None,
                hess[idx_G]  if hess  is not None else None,
                third[idx_G] if third is not None else None,
                params["weights"], n_G)

    lambda_reg = params["lambda_reg"]
    n = A.shape[0]
    if params["loss_type"] == "qp":
        coeffs, feas, rank_p1 = solve_qp(A, yH_train, G, params["qp_margin"])
        mask = np.ones(rank_p1, bool)
        if not feas: print("      [QP] INFAISABLE : ce dico ne sépare pas avec la marge")
    elif params["loss_type"] == "sigmoid":
        coeffs, n_mrg, rank_p1 = solve_sigmoid(A, yH_train, G, lambda_reg, BETA_SIG)
        mask = np.ones(rank_p1, bool)
    elif params["loss_type"] == "hinge2":
        coeffs, n_act, rank_p1 = solve_hinge2(A, yH_train, G, lambda_reg, params["a_label"])
        mask = np.ones(rank_p1, bool)
    elif params["loss_type"] == "logcosh":
        coeffs, rank_p1 = solve_logcosh(A, yH_train, G, lambda_reg, params["beta"])
        mask = np.ones(rank_p1, bool)
    else:
        M   = (A.T@A)/n + lambda_reg*G
        rhs = (A.T@yH_train)/n
        eigvals, eigvecs = np.linalg.eigh(M)
        mask = eigvals > max(lambda_reg, 0)
        V = eigvecs[:,mask]; S = eigvals[mask]; coeffs = V@((V.T@rhs)/S)

    F_train[:,slot] = A     @ coeffs
    F_test [:,slot] = Atest @ coeffs
    F_all  [:,slot] = Aall  @ coeffs

    fi_derivs_list.append({
        'phi':   phi   @ coeffs,
        'grad':  np.einsum('xjk,j->xk',    grad, coeffs) if grad  is not None else None,
        'hess':  np.einsum('xjkl,j->xkl',  hess, coeffs) if hess  is not None else None,
        'third': np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None,
    })

    data_i, reg_i, acc_tr_i, acc_te_i = method_metrics(
        F_train[:,slot], F_test[:,slot], coeffs, G, lambda_reg, yH_train, yH_test)
    loss_total = data_i + reg_i
    t1 = time.time(); times_p1.append(t1-t0)
    print(f"  {expert_names[slot]:7s} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_total:.6f} (data={data_i:.6f} reg={reg_i:.6f}) | "
          f"acc_tr={acc_tr_i:.2f} acc_te={acc_te_i:.3f} | "
          f"σ∈[{sigmas_sel.min():.2f},{sigmas_sel.max():.2f}] | {times_p1[-1]:.1f}s")

# ══════════════════════════════════════════════════════════════════════════════
# EXPERT POLYNOMIAL (optionnel : deg_poly_expert >= 0)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import PolynomialFeatures

USE_POLY_EXPERT = params["deg_poly_expert"] >= 0
print("\nCalibration expert polynomial..." if USE_POLY_EXPERT
      else "\nExpert polynomial DÉSACTIVÉ (deg_poly_expert < 0)")

rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params["n_G_H"]), replace=False)
n_H   = len(idx_H)
X_H   = X_all[idx_H]

w1 = params["weights"].get(1, 0.)
w2 = params["weights"].get(2, 0.)
w3 = params["weights"].get(3, 0.)

if USE_POLY_EXPERT:
    deg_e   = params["deg_poly_expert"]
    d_      = params["n_ambiant"]

    # ── Normalisation du domaine [lo,hi] -> [-1,1] pour le CONDITIONNEMENT ──────
    # À degré 13 sur [0,50], x^13 ~ 1e22 : la base monomiale est numériquement
    # morte. On travaille en u = (x-lo)/(hi-lo)*2 - 1 ∈ [-1,1]. Les dérivées ∂/∂x
    # portent alors un facteur SCALE = 2/(hi-lo) par ordre, appliqué pour que la
    # norme Sobolev du polynôme soit dans les MÊMES coordonnées x que les noyaux.
    SCALE = 2.0 / (DOM_HI - DOM_LO)
    def to_u(X): return (X - DOM_LO) * SCALE - 1.0

    poly_e  = PolynomialFeatures(degree=deg_e, include_bias=False)
    Phi_tr  = poly_e.fit_transform(to_u(X_train))     # features en coord. normalisées
    powers_e = poly_e.powers_
    n_feat  = Phi_tr.shape[1]

    def poly_deriv_eval_u(U, powers, coefs, dims=()):
        """Dérivée ∂/∂u_dims du polynôme(coefs) évalué en U (coord. normalisées)."""
        P = powers.astype(np.float64).copy()
        mult = coefs.astype(np.float64).copy()
        for m in dims:
            mult = mult * P[:, m]
            P[:, m] -= 1
        valid = mult != 0
        if not np.any(valid):
            return np.zeros(len(U))
        Up = U[:, None, :] ** P[None, valid, :]
        return Up.prod(axis=2) @ mult[valid]

    # ── Construire la Gram Sobolev G_poly du dictionnaire polynomial ───────────
    # sur les n_G points de Gram (comme les noyaux). Base = colonnes de Phi.
    # Pour chaque feature j (coef = e_j), on évalue ses dérivées ∂/∂x = SCALE^o ∂/∂u.
    rngP   = np.random.default_rng(seed=4242)
    n_Gp   = min(N_all, params["n_G"])
    idx_Gp = rngP.choice(N_all, size=n_Gp, replace=False)
    U_G    = to_u(X_all[idx_Gp])

    # tenseurs (n_Gp, n_feat, [d]^o) des dérivées de CHAQUE feature, en coord. x
    E = np.eye(n_feat)
    PHI_G  = poly_e.transform(U_G)                              # (n_Gp, n_feat)
    GRAD_G = None; HESS_G = None; THIRD_G = None
    if w1:
        GG = np.zeros((n_Gp, n_feat, d_))
        for j in range(n_feat):
            for a in range(d_):
                GG[:, j, a] = SCALE * poly_deriv_eval_u(U_G, powers_e, E[j], (a,))
        GRAD_G = GG
    if w2:
        HG = np.zeros((n_Gp, n_feat, d_, d_))
        for j in range(n_feat):
            for a in range(d_):
                for b in range(a, d_):
                    v = SCALE**2 * poly_deriv_eval_u(U_G, powers_e, E[j], (a,b))
                    HG[:,j,a,b] = v; HG[:,j,b,a] = v
        HESS_G = HG
    if w3:
        from itertools import permutations as _perm
        TG = np.zeros((n_Gp, n_feat, d_, d_, d_))
        for j in range(n_feat):
            for a in range(d_):
                for b in range(a, d_):
                    for cc in range(b, d_):
                        v = SCALE**3 * poly_deriv_eval_u(U_G, powers_e, E[j], (a,b,cc))
                        for p in set(_perm((a,b,cc))):
                            TG[:,j,p[0],p[1],p[2]] = v
        THIRD_G = TG

    G_poly = build_G(PHI_G, GRAD_G, HESS_G, THIRD_G, params["weights"], n_Gp)  # (n_feat,n_feat)

    # ── Fit sous la MÊME loss que loss_type (ici sigmoïde), pénalité G_poly ────
    lambda_reg = params["lambda_reg"]
    if params["loss_type"] == "qp":
        # QP dans la BASE H-ORTHONORMÉE, avec CENTRAGE sur le cloud + constante libre.
        # (1) On centre les features : φ̃ = φ − mean_cloud(φ). La direction constante
        #     sort ainsi du span des features centrées (elle a norme H nulle, elle
        #     polluait le plongement spectral et était jetée par le seuil).
        # (2) On ajoute explicitement la colonne 1 comme direction LIBRE (non pénalisée,
        #     norme H = 0) : elle cale le niveau de décision (seuil sign(u), 0) sans
        #     coût. Le plateau, qui avait besoin d'un offset constant, devient
        #     atteignable à petite norme.
        #   min ‖α‖²  s.c.  diag(y)[ (Φ̃_tr T) α + β·1 ] >= marge   (β libre)
        mean_phi = PHI_G.mean(axis=0)                  # moyenne des features sur le cloud
        Phi_tr_c = Phi_tr - mean_phi                   # features centrées (train)
        # Gram Sobolev inchangée (le centrage ne change pas les dérivées), on la ré-ortho
        s_g, V_g = np.linalg.eigh(G_poly)
        keep = (s_g >  params["thres1"]) & (s_g <  params["thres2"])
        Tmat = V_g[:, keep] / np.sqrt(s_g[keep])
        Phi_tr_o = Phi_tr_c @ Tmat                     # (n_train, r) centré + ortho
        r_o = int(keep.sum())
        # colonne constante libre : on l'ajoute comme dernière variable non pénalisée
        A_qp = np.hstack([Phi_tr_o, np.ones((len(Phi_tr_o), 1))])   # (n, r+1)
        G_qp = np.eye(r_o + 1); G_qp[-1, -1] = 0.0     # constante : norme H nulle -> libre
        sol, feas_p, rank_p = solve_qp(A_qp, yH_train, G_qp, params["qp_margin"], verbose=True)
        alpha_o = sol[:r_o]; beta_const = sol[-1]
        # reconstruire coef monomial : u(x) = φ̃(x)·(T α) + β = φ(x)·(Tα) + (β − mean_phi·Tα)
        coef_poly = Tmat @ alpha_o
        const_offset = beta_const - mean_phi @ coef_poly
        print(f"      [QP poly] base H-ortho rang {r_o}/{n_feat}, "
              f"max|α|={np.abs(alpha_o).max():.2e}, offset={const_offset:.3f}, faisable={feas_p}")
        # injecter l'offset constant dans les prédictions via un intercept
        _poly_intercept = const_offset
    elif params["loss_type"] == "sigmoid":
        coef_poly, n_mrg_p, rank_p = solve_sigmoid(Phi_tr, yH_train, G_poly, lambda_reg, BETA_SIG)
    elif params["loss_type"] == "hinge2":
        coef_poly, n_act_p, rank_p = solve_hinge2(Phi_tr, yH_train, G_poly, lambda_reg, params["a_label"])
    elif params["loss_type"] == "logcosh":
        coef_poly, rank_p = solve_logcosh(Phi_tr, yH_train, G_poly, lambda_reg, params["beta"])
    else:
        n = Phi_tr.shape[0]
        Mp = (Phi_tr.T@Phi_tr)/n + lambda_reg*G_poly
        rhsp = (Phi_tr.T@yH_train)/n
        evp, evecp = np.linalg.eigh(Mp); mkp = evp > max(lambda_reg,1e-12)
        coef_poly = evecp[:,mkp] @ ((evecp[:,mkp].T @ rhsp)/evp[mkp]); rank_p = int(mkp.sum())

    pred_poly_train = Phi_tr @ coef_poly
    pred_poly_test  = poly_e.transform(to_u(X_test)) @ coef_poly

    # ── Dérivées du polynôme AJUSTÉ sur X_H (coord. x) pour la Gram de Phase 2 ──
    U_H = to_u(X_H)
    delta_phi_poly = poly_e.transform(U_H) @ coef_poly
    if w1 or w2 or w3:
        delta_grad_poly = SCALE * np.stack(
            [poly_deriv_eval_u(U_H, powers_e, coef_poly, (a,)) for a in range(d_)], axis=1)
    else:
        delta_grad_poly = None
    if w2 or w3:
        delta_hess_poly = np.zeros((n_H, d_, d_))
        for a in range(d_):
            for b in range(a, d_):
                v = SCALE**2 * poly_deriv_eval_u(U_H, powers_e, coef_poly, (a,b))
                delta_hess_poly[:,a,b] = v; delta_hess_poly[:,b,a] = v
    else:
        delta_hess_poly = None
    if w3:
        from itertools import permutations
        delta_third_poly = np.zeros((n_H, d_, d_, d_))
        for a in range(d_):
            for b in range(a, d_):
                for cc in range(b, d_):
                    v = SCALE**3 * poly_deriv_eval_u(U_H, powers_e, coef_poly, (a,b,cc))
                    for p in set(permutations((a,b,cc))):
                        delta_third_poly[:,p[0],p[1],p[2]] = v
    else:
        delta_third_poly = None

    data_p, reg_p, acc_tr_p, acc_te_p = method_metrics(
        pred_poly_train, pred_poly_test, coef_poly, G_poly, lambda_reg, yH_train, yH_test)
    print(f"  Poly{deg_e}  | feat={n_feat:4d} rang={rank_p:4d} | "
          f"loss={data_p+reg_p:.6f} (data={data_p:.6f} reg={reg_p:.6f}) | "
          f"acc_tr={acc_tr_p:.2f} acc_te={acc_te_p:.3f}")

    F_train = np.column_stack([F_train, pred_poly_train])
    F_test  = np.column_stack([F_test,  pred_poly_test])
    fi_derivs_list.append({
        'phi': delta_phi_poly,   'grad': delta_grad_poly,
        'hess': delta_hess_poly, 'third': delta_third_poly})
    expert_names += [f"Poly{deg_e}"]

print(f"  H étendu à {len(fi_derivs_list)} fonctions")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : Gram finale sur H  (G_H_{ij} = <f_i, f_j>_Sobolev sur idx_H)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram finale sur H ({len(fi_derivs_list)} fonctions, n_G_H={n_H})...")

n_lev = len(fi_derivs_list)
G_H = np.zeros((n_lev, n_lev))
w0 = params["weights"].get(0,0.); w1 = params["weights"].get(1,0.)
w2 = params["weights"].get(2,0.); w3 = params["weights"].get(3,0.)

if w0:
    PHI_H = np.stack([
        fi['phi'] if fi['phi'].shape[0] == n_H else fi['phi'][idx_H]
        for fi in fi_derivs_list], axis=1)
    G_H += w0*(PHI_H.T@PHI_H)/n_H

if w1:
    idx_with_grad = [i for i, fi in enumerate(fi_derivs_list) if fi['grad'] is not None]
    if idx_with_grad:
        GRAD_H = np.stack([
            (fi_derivs_list[i]['grad'] if fi_derivs_list[i]['grad'].shape[0] == n_H
             else fi_derivs_list[i]['grad'][idx_H])
            for i in idx_with_grad], axis=1)
        G_part = np.einsum('xik,xjk->ij', GRAD_H, GRAD_H, optimize='optimal') * w1 / n_H
        G_H[np.ix_(idx_with_grad, idx_with_grad)] += G_part

if w2:
    idx_with_hess = [i for i, fi in enumerate(fi_derivs_list) if fi['hess'] is not None]
    if idx_with_hess:
        HESS_H = np.stack([
            (fi_derivs_list[i]['hess'] if fi_derivs_list[i]['hess'].shape[0] == n_H
             else fi_derivs_list[i]['hess'][idx_H])
            for i in idx_with_hess], axis=1)
        G_part = np.einsum('xikl,xjkl->ij', HESS_H, HESS_H, optimize='optimal') * w2 / n_H
        G_H[np.ix_(idx_with_hess, idx_with_hess)] += G_part

if w3:
    idx_with_third = [i for i, fi in enumerate(fi_derivs_list) if fi['third'] is not None]
    if idx_with_third:
        thirds = []
        for i in idx_with_third:
            t = fi_derivs_list[i]['third']
            thirds.append(t if t.shape[0] == n_H else t[idx_H])
        THIRD_H = np.stack(thirds, axis=1)
        G_partial = np.einsum('xiklm,xjklm->ij', THIRD_H, THIRD_H, optimize='optimal') * w3 / n_H
        for ii, i in enumerate(idx_with_third):
            for jj, j in enumerate(idx_with_third):
                G_H[i, j] += G_partial[ii, jj]

print(f"  G_H calculée ({n_H} pts)")

# ── SÉLECTION DES EXPERTS ─────────────────────────────────────────────────────
# En mode QP (semi-interpolation) il n'y a PAS de lambda : la norme H est
# l'objectif, pas une pénalité, et les contraintes de marge remplacent le terme
# de données. Le critère MSE-régularisé ci-dessous (hérité du mode mse) n'a donc
# aucun sens en QP -> on garde TOUS les experts et on laisse le QP de Phase 2
# leur attribuer leurs poids (un expert inutile reçoit un poids ~0, puisque le QP
# minimise la norme H sous contraintes).
n_loc  = len(y_train)
if params["loss_type"] == "qp":
    expert_losses = np.full(n_lev, np.nan)
    c_opt = np.full(n_lev, np.nan)
    sel = np.ones(n_lev, bool)
    sel_idx = np.arange(n_lev)
    print(f"\n  Sélection experts : DÉSACTIVÉE en mode QP (pas de lambda) "
          f"-> {n_lev}/{n_lev} experts gardés, le QP pondère.")
else:
    m_vec  = F_train.T @ yH_train / n_loc
    q_vec  = np.sum(F_train**2, axis=0) / n_loc
    denom  = q_vec + params["lambda_reg"] * np.diag(G_H)
    c_opt  = m_vec / denom
    expert_losses = np.mean(yH_train**2) - m_vec**2 / denom
    best_loss = expert_losses.min()
    sel       = expert_losses <= params["k_loss"] * best_loss
    sel_idx   = np.where(sel)[0]

    print(f"\n  Sélection experts (k_loss={params['k_loss']}, "
          f"seuil={params['k_loss']*best_loss:.3e}) :")
    for i in range(n_lev):
        tag = "GARDÉ " if sel[i] else "écarté"
        print(f"    {expert_names[i]:7s} : loss={expert_losses[i]:.3e}  c*={c_opt[i]:+.3f}  [{tag}]")
    print(f"  -> {sel.sum()}/{n_lev} experts retenus")

F_train_sel = F_train[:, sel]
F_test_sel  = F_test[:,  sel]
G_H_sel     = G_H[np.ix_(sel_idx, sel_idx)]

n_tr = F_train_sel.shape[0]
if params["loss_type"] == "qp":
    alpha_sel, feas_H, rank_H = solve_qp(F_train_sel, yH_train, G_H_sel,
                                         params["qp_margin"], verbose=True)
    mask_H = np.ones(rank_H, bool)
    print(f"  QP : faisable={feas_H}  ({rank_H} contraintes actives)")
elif params["loss_type"] == "sigmoid":
    alpha_sel, n_mrg_H, rank_H = solve_sigmoid(F_train_sel, yH_train, G_H_sel,
                                               params["lambda_reg"], BETA_SIG,
                                               verbose=True)
    mask_H = np.ones(rank_H, bool)
    print(f"  sigmoïde : {n_mrg_H}/{len(y_train)} points encore dans la marge")
elif params["loss_type"] == "hinge2":
    alpha_sel, n_act_H, rank_H = solve_hinge2(F_train_sel, yH_train, G_H_sel,
                                              params["lambda_reg"], params["a_label"],
                                              verbose=True)
    mask_H = np.ones(rank_H, bool)
    print(f"  hinge² : {n_act_H}/{len(y_train)} points actifs (marge={params['a_label']:g})")
elif params["loss_type"] == "logcosh":
    alpha_sel, rank_H = solve_logcosh(F_train_sel, yH_train, G_H_sel,
                                      params["lambda_reg"], params["beta"],
                                      verbose=False)
    mask_H = np.ones(rank_H, bool)
else:
    M_H  = (F_train_sel.T@F_train_sel)/n_tr + params["lambda_reg"]*G_H_sel
    rhs_H = (F_train_sel.T@yH_train)/n_tr
    eigvals_H, eigvecs_H = np.linalg.eigh(M_H)
    mask_H = eigvals_H > 1e-10
    V_H = eigvecs_H[:,mask_H]; S_H = eigvals_H[mask_H]
    alpha_sel = V_H@((V_H.T@rhs_H)/S_H)

alpha = np.zeros(n_lev); alpha[sel_idx] = alpha_sel
pred_H_train = F_train_sel @ alpha_sel
pred_H_test  = F_test_sel  @ alpha_sel
_, _, accH_tr, accH_te = method_metrics(pred_H_train, pred_H_test, None,
                                        np.zeros((1,1)), 0.0, yH_train, yH_test)
mse_H = mean_squared_error(yH_test, pred_H_test)          # gardé pour info
mse_H_train = mean_squared_error(yH_train, pred_H_train)

print(f"  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_lev} experts)")
print(f"  accuracy train = {accH_tr:.3f}  accuracy test = {accH_te:.3f}"
      f"   (MSE test={mse_H:.4f}, échelle {'±1' if params['loss_type']=='sigmoid' else '±a'})")
print(f"  alpha = {alpha.round(3)}")

# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV

print("\nCalibration RBF (baseline de comparaison uniquement, hors H)...")
gamma_grid = np.logspace(-3,2,20)
param_grid = {'alpha':[1e-8,1e-5,1e-3],'gamma':gamma_grid,'kernel':['rbf']}
kr_cv = GridSearchCV(KernelRidge(), param_grid=param_grid, cv=5,
                     scoring='neg_mean_squared_error', n_jobs=-1)
kr_cv.fit(X_train, y_train_pm1)
best_gamma = kr_cv.best_params_['gamma']
best_err_rbf = mean_squared_error(y_test_pm1, kr_cv.predict(X_test))
results = kr_cv.cv_results_; order = np.argsort(results['rank_test_score'])
print("\nTop 3 RBF :")
for i in range(3):
    idx = order[i]; g = results['param_gamma'][idx]; a = results['param_alpha'][idx]
    kr = KernelRidge(kernel='rbf', gamma=float(g), alpha=float(a))
    kr.fit(X_train, y_train_pm1)
    print(f"  #{i+1} γ={g:.4f} α={a:.0e} TEST={mean_squared_error(y_test_pm1, kr.predict(X_test)):.6f}")

poly = PolynomialFeatures(degree=min(params["deg_P"],8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8); ridge_poly.fit(poly.fit_transform(X_train), y_train_pm1)
err_poly = mean_squared_error(y_test_pm1, ridge_poly.predict(poly.transform(X_test)))

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*70); print("RÉSUMÉ"); print("="*70)
print(f"Polynomial Ridge  : {err_poly:.6f}")
print(f"RBF (γ={best_gamma:.4f})  : {best_err_rbf:.6f}")
print(f"\nAccuracy test individuelle des experts :")
for i in range(n_lev):
    acc_te_i = np.mean(np.sign(F_test[:,i]) == np.sign(yH_test))
    tag = "" if sel[i] else "  (écarté)"
    _lstr = "n/a (QP)" if np.isnan(expert_losses[i]) else f"{expert_losses[i]:.3e}"
    print(f"  {expert_names[i]:7s} : acc_te={acc_te_i:.3f}  loss={_lstr}{tag}")
print(f"\nSobolev sous-espace H : accuracy test = {accH_te:.3f}  (rang={mask_H.sum()})")
print(f"\nweights={params['weights']}  λ={params['lambda_reg']} (unique)")
print(f"n_levels={params['n_levels']} (+{params['n_levels_wnd']} wnd, "
      f"poly={'OFF' if not USE_POLY_EXPERT else params['deg_poly_expert']})  "
      f"n_centres={params['n_centres']}  n_dict={params['n_dict']}  "
      f"k_loss={params['k_loss']} ({sel.sum()}/{n_lev} experts retenus)")
print(f"ε={params['epsilon']}  σ∈[{params['sigma_min']},{params['sigma_max']}] (sans decay)")
print(f"temps phase 1 : {sum(times_p1):.1f}s total  ({np.mean(times_p1):.1f}s/niveau)")

from sklearn.metrics import roc_auc_score, accuracy_score
# Décision par sign(f) : seuil 0. Labels de test en {0,1} pour les métriques.
y_test_class = (y_test > 0).astype(int)
pred_H     = F_test  @ alpha
pred_rbf   = kr_cv.predict(X_test)
pred_ridge = ridge_poly.predict(poly.transform(X_test))
print("\n" + "="*70)
print("CLASSIFICATION (décision sign(f), seuil = 0)")
print("="*70)
for nom, pred in [("Sobolev H", pred_H), ("RBF", pred_rbf), ("Ridge poly", pred_ridge)]:
    pred_class = (pred > 0).astype(int)
    acc = accuracy_score(y_test_class, pred_class)
    try:    auc = roc_auc_score(y_test_class, pred)
    except Exception: auc = float('nan')
    print(f"  {nom:12s} : accuracy={acc:.4f}  AUC={auc:.4f}")


Génération du cloud sur {Q=0} = {P_sep=0} ∪ {P_sep=ε} (projection exacte, 50/50)...
  X_train:(10, 4)  X_test:(5000, 4)  X_unlabeled:(3990, 4)
  train : plan0=5  planε=5  max|résidu projection|=4.22e-15
  test  : plan0=2500  planε=2500  max|résidu projection|=4.66e-15
  y_train: 5 × (−1)  |  5 × (+1)   (labels ±a_label, décision sign(f))

Phase 1 : calcul de 0 experts (0 cyl + 0 wnd C2)...

Calibration expert polynomial...
      QP: succès=True marge_atteinte=1.000 ||u||_H=0.0784
      [QP poly] base H-ortho rang 700/714, max|α|=4.91e-02, offset=0.012, faisable=True
  Poly9  | feat= 714 rang=   7 | loss=106.732179 (data=0.025322 reg=106.706857) | acc_tr=1.00 acc_te=1.000
  H étendu à 1 fonctions

Phase 2 : Gram finale sur H (1 fonctions, n_G_H=4000)...
  G_H calculée (4000 pts)

  Sélection experts : DÉSACTIVÉE en mode QP (pas de lambda) -> 1/1 experts gardés, le QP pondère.
      QP: succès=True marge_atteinte=1.000 ||u||_H=0.2357
  QP : faisable=True  (4 contraintes actives)
  rang H

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
